# Tokens and Context Windows

This notebook is a detailed interview-preparation reference for tokenization and context management in LLM systems. It expands from first principles to practical production trade-offs in cost, latency, chunking, and retrieval.

If you can explain this notebook clearly, you can usually handle both conceptual and applied interview rounds about prompt engineering and LLM system design.

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

np.random.seed(42)
np.set_printoptions(precision=3, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")

---

## 1. What Is a Token?

A token is the basic discrete unit an LLM processes. It is not always a word. Depending on the tokenizer, a token may be a full word, a subword fragment, punctuation, whitespace pattern, or even byte-derived piece. The model reads token IDs, not raw strings.

Understanding tokens matters because every major operational metric scales with token count: context usage, latency, and API cost. Interviewers frequently ask this question early, and a strong answer ties tokenization directly to system performance.

---

## 2. Why Modern Tokenizers Use Subwords

Word-level vocabularies explode in size and still fail on rare or new terms. Character-level tokenization avoids out-of-vocabulary issues but creates long sequences that increase compute cost. Subword methods strike a practical balance by reusing frequent fragments while still covering rare words compositionally.

This balance improves memory efficiency and generalization in multilingual or domain-heavy corpora. In interviews, emphasize the trade-off triangle: vocabulary size, sequence length, and robustness to unseen text.

---

## 3. BPE, WordPiece, and SentencePiece

Byte Pair Encoding (BPE) starts from small symbols and repeatedly merges frequent adjacent pairs. WordPiece uses a similar idea but optimizes likelihood criteria during vocabulary growth. SentencePiece can operate without whitespace assumptions, which is useful for multilingual text and script diversity.

A practical interview comparison: BPE is frequency-merge oriented, WordPiece is likelihood-oriented, SentencePiece is tokenizer-framework oriented and language-agnostic in segmentation assumptions. All are subword strategies with different optimization details.

In [ ]:
def simple_tokenize(text):
    return re.findall(r"\w+|[^\w\s]", text.lower())

def rough_token_count(text):
    # quick planning heuristic for English prose
    return max(1, len(text) // 4)

samples = [
    "Tokenization affects both quality and cost.",
    "Short prompts are usually cheaper and faster.",
    "Subword vocabularies balance compression and coverage.",
]

for s in samples:
    print(s)
    print("chars:", len(s), "rough_tokens:", rough_token_count(s), "regex_tokens:", len(simple_tokenize(s)))
    print()

---

## 4. BPE Merge Process Intuition

BPE-style training repeatedly identifies the most frequent adjacent pair and merges it into a new symbol. Over many iterations, common fragments become single tokens, reducing sequence lengths on average. Rare strings remain decomposable into smaller units.

Interviewers may ask why this helps LLM quality. Shorter sequences improve compute efficiency for a fixed context window, and shared subword components improve representation reuse across related words.

In [ ]:
# Tiny BPE merge demonstration (conceptual)
corpus = ["low", "lowest", "newer", "wider"]
symbols = [list(w) + ["</w>"] for w in corpus]

def pair_counts(symbol_sequences):
    counts = Counter()
    for seq in symbol_sequences:
        for i in range(len(seq) - 1):
            counts[(seq[i], seq[i+1])] += 1
    return counts

def merge_pair(symbol_sequences, pair):
    a, b = pair
    merged = []
    for seq in symbol_sequences:
        new_seq = []
        i = 0
        while i < len(seq):
            if i < len(seq) - 1 and seq[i] == a and seq[i+1] == b:
                new_seq.append(a + b)
                i += 2
            else:
                new_seq.append(seq[i])
                i += 1
        merged.append(new_seq)
    return merged

for step in range(4):
    counts = pair_counts(symbols)
    best_pair, best_count = counts.most_common(1)[0]
    print(f"Step {step+1}: merge {best_pair} (count={best_count})")
    symbols = merge_pair(symbols, best_pair)

print("Final symbolized corpus:", symbols)

---

## 5. Token Counting and the tiktoken Concept

In production, rough character-based estimates are useful for planning, but exact counts must use the same tokenizer as the serving model. Different models can tokenize the same string differently, especially for code, punctuation-heavy text, and non-English scripts.

When interviewers mention tools like tiktoken, the important concept is deterministic model-specific token counting for budgeting. The key skill is not memorizing one library API; it is understanding tokenizer-model coupling.

---

## 6. Context Windows and Budget Equation

The context window is the maximum token capacity the model can process in one forward pass. This budget includes system prompts, user input, retrieval chunks, tool outputs, and generated completion. A standard planning equation is:

\[
T_{system} + T_{input} + T_{retrieval} + T_{output} \leq T_{window}
\]

In interviews, mention reserved output budgets. If you consume the full window with input, generation truncates or fails. Good systems allocate output room first, then build input context under that constraint.

In [ ]:
def context_budget(window_size, prompt_tokens, reserve_output=0):
    return max(0, window_size - prompt_tokens - reserve_output)

window = 8192
prompt_options = [500, 1200, 2600, 5000, 7600]
reserve = 300

for p in prompt_options:
    print(f"window={window}, prompt={p}, reserve={reserve} -> max_output={context_budget(window, p, reserve)}")

---

## 7. Context Window Comparison Across Model Tiers

Different model families expose different context limits, and larger windows can improve long-document handling but may increase cost or latency. Practical system design uses routing so that short tasks run on smaller windows and only long tasks trigger expensive long-context models.

A conceptual comparison table for interview discussion:

| Model Tier (Conceptual) | Typical Context Window | Best Fit |
|---|---:|---|
| Small assistant tier | 8K-16K | Chat, short docs, fast workflows |
| Mid reasoning tier | 32K-64K | Multi-document synthesis |
| Long-context tier | 100K+ | Long reports, legal or research archives |

The interview signal is decision reasoning: choose the smallest window that reliably solves the task.

---

## 8. Chunking Strategies for Long Documents

Fixed-size chunking is simple and fast, but can split ideas across boundaries. Sentence-based chunking preserves language units better but may produce uneven lengths. Semantic chunking groups content by meaning similarity, often improving retrieval precision at higher preprocessing complexity.

No single strategy is universally best. Effective implementations test chunk quality against task metrics: answer accuracy, citation quality, latency, and index size. Interview answers should include this metric-driven selection mindset.

In [ ]:
def fixed_chunk(tokens, chunk_size=12, overlap=3):
    out = []
    step = max(1, chunk_size - overlap)
    for i in range(0, len(tokens), step):
        c = tokens[i:i+chunk_size]
        if not c:
            break
        out.append(c)
        if i + chunk_size >= len(tokens):
            break
    return out

text = "Context windows constrain what a model can see. Chunking with overlap preserves nearby context and improves retrieval reliability in many workloads."
tokens = simple_tokenize(text)
chunks = fixed_chunk(tokens, chunk_size=10, overlap=2)

for i, c in enumerate(chunks, 1):
    print(f"Chunk {i}: {c}")

---

## 9. RAG Chunk Size Trade-Offs

In retrieval-augmented generation (RAG), chunk size affects both retrieval precision and synthesis completeness. Smaller chunks improve retrieval granularity but may lose context needed for final answers. Larger chunks preserve context but can dilute relevance and waste token budget.

A practical approach is to tune chunk size and overlap jointly, then evaluate end-to-end answer quality. Interviewers appreciate candidates who mention retrieval metrics (recall@k, hit rate) and downstream answer metrics together rather than optimizing one in isolation.

---

## 10. Cost and Latency Formulas

Approximate cost can be modeled as linear in token volume:

\[
\text{Cost} \approx c_{in} \cdot T_{in} + c_{out} \cdot T_{out}
\]

Latency can be decomposed into fixed overhead plus per-token generation cost:

\[
\text{Latency} \approx t_0 + \alpha \cdot T_{in} + \beta \cdot T_{out}
\]

These formulas are simplified, but useful for capacity planning and architecture comparisons.

In [ ]:
# Conceptual cost and latency calculator
in_tokens = np.array([300, 700, 1500, 3200])
out_tokens = np.array([120, 220, 400, 700])

c_in = 0.0000015
c_out = 0.0000030
cost = c_in * in_tokens + c_out * out_tokens

latency = 0.25 + 0.00004 * in_tokens + 0.00009 * out_tokens

for i, o, c, l in zip(in_tokens, out_tokens, cost, latency):
    print(f"in={i:4d}, out={o:4d} -> cost=${c:.4f}, latency~{l:.3f}s")

plt.figure(figsize=(7, 4))
plt.plot(in_tokens + out_tokens, latency, marker="o", label="Latency")
plt.plot(in_tokens + out_tokens, cost * 300, marker="s", label="Scaled Cost")
plt.xlabel("Total tokens")
plt.ylabel("Relative scale")
plt.title("Token Volume vs Latency/Cost (conceptual)")
plt.legend()
plt.show()

---

## 11. Padding, Truncation, and Batching

Batch inference often requires equal sequence lengths. Short sequences are padded; long sequences may be truncated. Attention masks ensure padded positions do not contribute to attention or loss. While conceptually straightforward, these details materially impact throughput and model quality.

In interviews, mention that truncation strategy should be task-aware. For summarization, dropping the beginning can be harmful. For chat, preserving recent turns may be more important than earliest turns.

In [ ]:
# Padding + truncation demo
seqs = [
    [11, 22, 17, 9],
    [6, 3, 18, 25, 31, 2, 7],
    [14, 5],
]

max_len = 6
pad_id = 0
batch = np.full((len(seqs), max_len), pad_id, dtype=int)
mask = np.zeros_like(batch)

for i, s in enumerate(seqs):
    clipped = s[:max_len]
    batch[i, :len(clipped)] = clipped
    mask[i, :len(clipped)] = 1

print("Batch:\n", batch)
print("Mask:\n", mask)

---

## 12. Practical Token Optimization Tactics

Token optimization starts with concise prompting and careful retrieval filtering. Remove repeated instructions, avoid unnecessary verbose context, and cache reusable prompt prefixes where possible. Structured outputs can reduce retries and improve deterministic parsing.

Model routing is another high-impact tactic: use smaller models for simple requests and escalate to larger models only when complexity demands it. Interviewers like hearing this because it shows production-level cost awareness.

---

## 13. Handling Long Documents Reliably

For long documents, common patterns include map-reduce summarization, hierarchical synthesis, and retrieval-first workflows. Sliding windows with overlap preserve local continuity, while hierarchical aggregation improves global coherence.

A robust implementation tracks citation coverage and unsupported claims. This makes long-context systems auditable and easier to trust in enterprise settings where source traceability matters.

---

## 14. Frequent Pitfalls in Token Budgeting

Common failures include forgetting system-prompt tokens, underestimating tool-output size, and not reserving completion budget. Another issue is chunk overlap that is too large, which inflates context and cost with minimal quality gain.

A practical mitigation is preflight budgeting: estimate token usage before each model call, then enforce hard limits with fallback behavior. In interviews, this demonstrates reliability thinking rather than ad-hoc prompt experimentation.

---

## 15. Interview Focus

## Interview Focus

### Q1. What is a token?
A token is the model's basic input unit, usually subword-level rather than whole-word-level. All context limits, cost, and latency are measured in tokens.

### Q2. Why not use word-level tokenization?
Word-level tokenization creates huge vocabularies and fails on unseen words. Subword tokenization balances coverage, vocabulary size, and sequence length.

### Q3. What is BPE in simple terms?
BPE repeatedly merges the most frequent adjacent symbol pairs, creating compact tokens for common patterns while preserving decomposition for rare strings.

### Q4. What is a context window?
It is the maximum number of tokens processed in a single model call, including system instructions, user input, retrieval context, and output.

### Q5. How do you handle inputs longer than context limits?
Use chunking, retrieval, hierarchical summarization, or sliding windows with overlap. Reserve output tokens to avoid truncation failures.

### Q6. How does chunk size affect RAG quality?
Small chunks improve retrieval precision but may lose context. Larger chunks preserve context but can reduce relevance density and increase cost.

### Q7. How do token costs scale?
Cost is approximately linear in input and output token counts with model-specific rates. Reducing unnecessary tokens often gives immediate savings.

### Q8. How do you optimize for latency?
Shorten prompts, cap outputs, cache reusable context, batch requests when possible, and route simple tasks to smaller models.

### Q9. What mistakes happen in token budgeting?
Teams often forget hidden prompt components and tool outputs, causing context overflow. Preflight estimation and hard guards prevent runtime failures.

---

## 16. Summary and Key Takeaways for Interviews

Tokenization and context budgeting are central to reliable LLM engineering. Understanding BPE/WordPiece/SentencePiece, accurate token counting, chunking design, and context allocation directly impacts quality, speed, and cost.

For interviews, be ready to explain both theory and execution: how tokens are created, how windows constrain architecture, and how to design robust long-document pipelines using chunking, retrieval, and explicit budget controls.